# **Technical Test**

## **2. Python**

<b>Would you be able to replicate the output of the SQL Question 8 in python pandas library? In that case, could you facilitate us the code?

You have to aggregate individual messages into conversations. The result should be used to create a table customer_courier_conversations (take into consideration that a conversation is unique per order).

a. The required fields are the following:

i. order_id

ii. city_code

iii. first_courier_message: Timestamp of the first courier message

iv. first_customer_message: Timestamp of the first customer message

v. num_messages_courier: Number of messages sent by courier

vi. num_messages_customer: Number of messages sent by customer

vii. first_message_by: The first message sender (courier or customer)

viii. conversation_started_at: Timestamp of the first message in the conversation

ix. first_responsetime_delay_seconds: Time (in secs) elapsed until the first message was responded

x. last_message_time: Timestamp of the last message sent

xi. last_message_order_stage: The stage of the order when the last message was sent</b>

In [2]:
import pandas as pd

mess = pd.read_csv('data/customer_courier_chat_messages.csv', delimiter=';')
ord = pd.read_csv('data/orders.csv', delimiter=';')

# mess.head()
# ord.head()

# mess.info()
mess['message_sent_time'] = pd.to_datetime(mess['message_sent_time'], format='%d/%m/%Y %H:%M')
# mess.info()

mess['sender'] = mess['sender_app_type'].str.split(' ').str[0]
# mess.head()

mess = mess.sort_values(['order_id', 'message_sent_time'])

# Group by order_id and sender
mess_agg = mess.groupby(['order_id', 'sender']).agg(
    first_message_time=('message_sent_time', 'min'),
    num_messages=('message_sent_time', 'count'),
    last_message_time=('message_sent_time', 'max'),
).sort_values('first_message_time').reset_index()
# mess_agg

#Courier Messages
cour_mess_agg = mess_agg[mess_agg['sender'] == 'Courier'].reset_index(drop=True)
cour_mess_agg = cour_mess_agg.rename(columns={'first_message_time': 'first_message_cour', 'num_messages': 'num_messages_cour', 'last_message_time': 'last_message_cour'})
# cour_mess_agg

#Customer Messages
cust_mess_agg = mess_agg[mess_agg['sender'] == 'Customer'].reset_index(drop=True)
cust_mess_agg = cust_mess_agg.rename(columns={'first_message_time': 'first_message_cust', 'num_messages': 'num_messages_cust', 'last_message_time': 'last_message_cust'})
# cust_mess_agg

total_mess_agg = mess.groupby('order_id').agg(
    first_message_time=('message_sent_time', 'min'),
    first_message_by=('sender', 'first'),
    last_message_time=('message_sent_time', 'max'),
    last_order_stage=('order_stage', 'last')
).sort_values('first_message_time').reset_index()
# total_mess_agg

#merge ord, cour_mess_agg, cust_mess_agg and total_mess_agg
convers = ord.merge(cour_mess_agg, on='order_id', how='left')
convers = convers.merge(cust_mess_agg, on='order_id', how='left')
convers = convers.merge(total_mess_agg, on='order_id', how='left')
# convers

convers_final = convers[['order_id', 'city_code', 'first_message_cour', 'num_messages_cour', 'first_message_cust', 'num_messages_cust', 'first_message_time', 'first_message_by',
                         'last_message_time', 'last_order_stage']].copy()

convers_final['responsedelay_secs'] = (convers_final['first_message_cust'] - convers_final['first_message_cour']).dt.total_seconds().abs()
# convers_final

convers_final['num_messages_cust'] = convers_final['num_messages_cust'].fillna(0).astype(int)
# convers_final

convers_final.rename(columns={'first_message_cour': 'first_courier_message', 'num_messages_cour': 'num_messages_courier', 'first_message_cust': 'first_customer_message',
                              'num_messages_cust': 'num_messages_customer', 'first_message_time': 'conversation_started_at', 'last_order_stage': 'last_message_order_stage',
                              'responsedelay_secs': 'first_responsetime_delay_seconds'}, inplace=True)

cols_order = ['order_id', 'city_code', 'first_courier_message', 'first_customer_message', 'num_messages_courier', 'num_messages_customer', 'first_message_by',
              'conversation_started_at', 'first_responsetime_delay_seconds', 'last_message_time', 'last_message_order_stage']
convers_final_ordered = convers_final[cols_order]
convers_final_ordered

,order_id,city_code,first_courier_message,first_customer_message,num_messages_courier,num_messages_customer,first_message_by,conversation_started_at,first_responsetime_delay_seconds,last_message_time,last_message_order_stage
0,38,BCN,2022-08-09 07:55:00,NaT,1,0,Courier,2022-08-09 07:55:00,NaN,2022-08-09 07:55:00,ADDRESS_DELIVERY
1,134,OPO,2022-08-07 10:01:00,2022-08-07 10:00:00,1,2,Customer,2022-08-07 10:00:00,60.0,2022-08-07 10:02:00,PICKING_UP
2,555,BCN,2022-08-09 08:01:00,2022-08-09 08:00:00,1,2,Customer,2022-08-09 08:00:00,60.0,2022-08-09 08:02:00,PICKING_UP
3,875,VAL,2022-08-07 14:50:00,2022-08-07 14:51:00,2,2,Courier,2022-08-07 14:50:00,60.0,2022-08-07 14:55:00,PICKING_UP
